# Fuselage Design — As-Selected Re-Solve (COTS Hardware)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB6 (`fuselage_design`) packaged the vehicle from mass-closure fractions
and *effective density estimates*. NB11 froze real hardware with catalog
masses and envelopes. This notebook **re-solves the fuselage with the
as-selected hardware** — downstream, as a new notebook, keeping the
pipeline one-way (ADR-0012). The conceptual `out/fuselage.yaml` remains
the geometry the CAD/CFD chain (NB8+) is built from; the deltas found here
are **findings** to fold back into `config/` in a deliberate next
iteration, never an automatic feedback edge.

Three things become real in this re-solve:

1. **Masses** — the COTS battery pack, FC + supporting avionics stack,
   motor + prop and ESC replace their fraction-derived allocations (the
   duct stays modeled: it is printed structure booked under propulsion).
2. **Packing** — the battery bay density comes from the frozen pack's own
   envelope, and each bay's length is **floored at the best-orientation
   axial length of the rigid part it must hold** (a volume-based bay
   assumes contents can be shaped to it; a rigid LiPo brick cannot).
3. **Fit** — every frozen part is checked against the space the re-solved
   hull actually gives it (reported, never filtered on).

## Axis Convention

Same as NB6: body FRD, stations from the nose tip +aft, $x_{body} = -x_s$.

## Inputs

- `out/components.yaml` *(NB11)* — frozen parts, envelopes, supporting stack
- `out/aileron_cots.yaml`, `out/vibration_cots.yaml` *(NB12/NB13)*
- `out/airfoil.yaml`, `out/control_vanes.yaml` *(NB2/NB3)*
- `out/fuselage.yaml` *(NB6 — conceptual solution, for the delta report)*
- `config/fuselage.yaml`, `config/modularity.yaml`

## Outputs

- `out/fuselage_cots.yaml` *(same schema as `out/fuselage.yaml` +
  `as_selected` / `bay_fit` / `delta_vs_conceptual` blocks)*
- `out/fuselage_layout_cots.png`

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from dataclasses import replace

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design import FrozenComponents, reports
from conceptual_design.plots import plot_fuselage_layout

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (same pattern as NB2–NB13) and load
the upstream handoffs: the as-selected aileron/vibration re-solves, the
frozen hardware list, and the conceptual fuselage for the delta report.

---

In [ ]:
# -- Re-run the sizing loop + freeze + as-selected upstream re-solves ----------
inp, result = solve_design_point(CONFIG_PATH)

af       = load_handoff(OUT_PATH, "airfoil")
vanes    = load_handoff(OUT_PATH, "control_vanes")
ail_cots = load_handoff(OUT_PATH, "aileron_cots")
vib_cots = load_handoff(OUT_PATH, "vibration_cots")
fus_est  = load_handoff(OUT_PATH, "fuselage")
comps    = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")

MTOW_kg      = result.m_total_kg
chord_mean_m = result.wing.chord_mean
S_wing       = result.wing.S_wing
R_hub_m      = vanes["R_hub_m"]
m_aileron_servo_kg   = ail_cots["servo_mass_kg_each"]   * ail_cots["n_ailerons"]
m_aileron_linkage_kg = ail_cots["linkage_mass_kg_each"] * ail_cots["n_ailerons"]
m_isolation_avionics_kg = vib_cots["m_isolation_avionics_kg"]
m_isolation_struct_kg   = vib_cots["m_isolation_struct_kg"]
sway_pad_m              = vib_cots["sway_pad_total_m"]

print(f"Converged MTOW (closure) : {MTOW_kg:.3f} kg")
print("Frozen hardware          : " + ", ".join(
    f"{comps[c].id}" for c in ("battery", "flight_controller", "esc",
                               "edf_motor", "propeller", "servo")))
print(f"Aileron hw (NB12)        : {(m_aileron_servo_kg + m_aileron_linkage_kg)*1e3:.1f} g")
print(f"Isolation hw (NB13)      : {(m_isolation_avionics_kg + m_isolation_struct_kg)*1e3:.1f} g, "
      f"sway pad {sway_pad_m*1e3:.2f} mm")

# Section 2 — As-Selected Masses and Envelopes

Replace the fraction-derived estimates with the frozen hardware
(`cots_integration`):

- **battery** — catalog mass, and packing density from its own envelope
  (mass / box volume) instead of the configured `rho_battery_pack`;
- **avionics** — the "fraction" is rebuilt bottom-up (FC + supporting
  stack + the servo/isolator carve-outs), so `size_fuselage`'s ADR-0005
  carve arithmetic nets back to the *actual* bay content;
- **propulsion** — actual motor + prop and ESC masses at their layout
  stations (duct stays modeled);
- **bay floors** — the battery pack and FC board envelopes must fit their
  bays in some orientation.

---

In [ ]:
from conceptual_design.fuselage_design import FuselageParams, ModularityParams
from conceptual_design.cots_integration import (
    avionics_budget_bottom_up, bay_part_envelopes, effective_density,
    propulsion_item_masses,
)

fus_p = FuselageParams.from_yaml(CONFIG_PATH / "fuselage.yaml")
mod_p = ModularityParams.from_yaml(CONFIG_PATH / "modularity.yaml")

battery = comps["battery"]
fc      = comps["flight_controller"]
servo   = comps["servo"]

rho_batt_eff = effective_density(battery)
m_avionics_cots = avionics_budget_bottom_up(
    m_fc_kg=fc.mass_kg, m_supporting_kg=comps.supporting_mass_kg,
    n_vane_servos=int(vanes["n_vanes"]), n_aileron_servos=int(ail_cots["n_ailerons"]),
    m_servo_each_kg=servo.mass_kg,
    m_isolation_avionics_kg=m_isolation_avionics_kg,
)
prop_masses = propulsion_item_masses(result.m_propulsion_kg, comps)
envelopes   = bay_part_envelopes(comps)

fus_p_cots = replace(fus_p,
                     rho_battery_pack=rho_batt_eff,
                     servo_mass_kg=servo.mass_kg)

print(f"{'quantity':<34}{'closure/est.':>14}{'as-selected':>14}")
print("-" * 62)
print(f"{'battery mass [g]':<34}{result.m_battery_kg*1e3:>14.1f}{battery.mass_kg*1e3:>14.1f}")
print(f"{'battery pack density [kg/m^3]':<34}{fus_p.rho_battery_pack:>14.0f}{rho_batt_eff:>14.0f}")
print(f"{'avionics budget [g]':<34}{result.m_avionics_kg*1e3:>14.1f}{m_avionics_cots*1e3:>14.1f}")
print(f"{'motor + prop [g]':<34}{0.60*result.m_propulsion_kg*1e3:>14.1f}{prop_masses['motor_fan']*1e3:>14.1f}")
print(f"{'ESC [g]':<34}{0.25*result.m_propulsion_kg*1e3:>14.1f}{prop_masses['esc']*1e3:>14.1f}")
print(f"{'vane/aileron servo, each [g]':<34}{fus_p.servo_mass_kg*1e3:>14.1f}{servo.mass_kg*1e3:>14.1f}")
print()
for bay, envs in envelopes.items():
    for shape, dims in envs:
        print(f"bay floor: {bay:<9} <- {shape} " +
              " x ".join(f"{d*1e3:.1f}" for d in dims) + " mm")

# Section 3 — Re-Solve the Hull

Same physics as NB6 (`size_fuselage`, one call) with the as-selected
masses, the envelope floors in the diameter/stack solve, and the actual
propulsion item masses in the layout. The wing is again placed for the
configured static margin **by construction** — what moves is everything
around it.

The as-selected all-up mass is the layout total: the closure MTOW plus
every standing mass finding of the freeze (battery quantisation, motor
class, avionics stack) in one number.

---

In [ ]:
from conceptual_design.fuselage_design import size_fuselage

fus = size_fuselage(
    m_battery_kg    = battery.mass_kg,
    m_payload_kg    = result.m_payload_kg,
    m_avionics_kg   = m_avionics_cots,
    m_propulsion_kg = result.m_propulsion_kg,
    m_structure_kg  = result.m_structure_kg,
    m_wing_kg       = result.m_wing_kg,
    m_misc_kg       = result.m_misc_kg,
    R_hub_m         = R_hub_m,
    D_rotor_m       = inp.rotor.D_rotor_m,
    c_vane_m        = vanes["c_vane_m"],
    n_vanes         = vanes["n_vanes"],
    S_vane_m2       = vanes["S_vane_m2"],
    hinge_xc        = vanes["hinge_xc"],
    chord_mean_m    = chord_mean_m,
    m_aileron_servo_kg   = m_aileron_servo_kg,
    m_aileron_linkage_kg = m_aileron_linkage_kg,
    m_isolation_avionics_kg = m_isolation_avionics_kg,
    m_isolation_struct_kg   = m_isolation_struct_kg,
    sway_pad_m              = sway_pad_m,
    V_cruise        = inp.mission.V_cruise,
    rho             = inp.env.rho,
    S_wing          = S_wing,
    p               = fus_p_cots,
    mod             = mod_p,
    construction_method = inp.wf.construction_method,
    part_envelopes   = envelopes,
    prop_item_masses = prop_masses,
)

m_asselected = sum(it.mass_kg for it in fus.items)
m_delta      = m_asselected - MTOW_kg

print(f"Active constraint : {fus.active_constraint.upper()}")
print(f"D_fus             : {fus.D_fus*1e3:7.1f} mm   (conceptual {fus_est['D_fus_m']*1e3:.1f} mm)")
print(f"L_fus             : {fus.L_fus*1e3:7.1f} mm   (conceptual {fus_est['L_fus_m']*1e3:.1f} mm)")
print(f"  nose / mid / tail : {fus.L_nose*1e3:.0f} / {fus.L_mid*1e3:.0f} / {fus.L_tail*1e3:.0f} mm")
print(f"Bay stack         : {fus.L_stack_m*1e3:7.1f} mm  of  {fus.L_avail_m*1e3:.1f} mm available "
      f"({fus.L_stack_m/fus.L_avail_m*100:.0f}% used)")
print(f"CG station        : {fus.x_CG*1e3:7.1f} mm   (conceptual {fus_est['x_CG_m']*1e3:.1f} mm)")
print(f"Structure ({fus.construction_method}) : {fus.m_shell_kg*1e3:.1f} g "
      f"of {fus.m_struct_budget_kg*1e3:.1f} g budget")
print()
print(f"As-selected all-up mass : {m_asselected:.3f} kg  "
      f"(closure MTOW {MTOW_kg:.3f} kg, {m_delta*1e3:+.0f} g of standing findings)")

# Section 4 — Physical Fit of the Frozen Hardware

Each frozen part against the space the re-solved hull gives it:
best-orientation axial length (+ clearance) vs. its bay, the ESC vs. its
mid-body slot, the motor's diameter vs. the tail centerbody. **Findings,
never filters** — same discipline as the NB11 mass-allocation report.

---

In [ ]:
from conceptual_design.cots_integration import bay_fit_report

fit = bay_fit_report(fus, fus_p_cots, comps)
reports.print_bay_fit_table(fit)

# Section 5 — As-Selected Internal Layout

Same side-view as NB6, drawn from the re-solved geometry. The battery bay
is now as long as the real pack; the CG marker is the as-selected one.

---

In [ ]:
plot_fuselage_layout(fus, fus_p_cots, vanes, chord_mean_m,
                     "As-selected fuselage layout — side view (COTS re-solve)",
                     OUT_PATH / "fuselage_layout_cots.png")

print(f"{'item':<14}{'mass [kg]':>10}{'x_cg [mm]':>12}")
for it in fus.items:
    print(f"{it.name:<14}{it.mass_kg:>10.3f}{it.x_cg*1e3:>12.1f}")
print("-" * 36)
print(f"{'total':<14}{m_asselected:>10.3f}{fus.x_CG*1e3:>12.1f}")

# Section 6 — Drag Budget and Vane Arm Re-Check

The hull that holds the real hardware is not the hull NB6 drew — its
wetted area and $C_{D0}$ move, and so does the CG-to-vane arm the NB3
sizing rests on. Re-check both against the same budgets as NB6.

---

In [ ]:
CD0_misc = 0.0025    # duct external + vanes + antennas + interference (as NB6)
CD0_total = af["Cd0_section"] * 1.10 + fus.CD0_fus + CD0_misc
drag_margin = (inp.aero.CD0 - CD0_total) / inp.aero.CD0 * 100

print(f"{'CD0 fuselage (as-selected)':<34}: {fus.CD0_fus:.5f}   (conceptual {fus_est['CD0_fus']:.5f})")
print(f"{'CD0 total (built up)':<34}: {CD0_total:.5f}")
print(f"{'CD0 assumed in sizing (NB1)':<34}: {inp.aero.CD0:.5f}")
print(f"Drag budget {'CLOSES' if CD0_total <= inp.aero.CD0 else 'EXCEEDED -- revisit aerodynamics.yaml'}"
      f"  (margin {drag_margin:+.1f}%)")
print()
L_assumed = vanes["L_CG_m"]
factor    = fus.L_vane_arm / L_assumed
print(f"{'Vane-to-CG arm (as-selected)':<34}: {fus.L_vane_arm*1e3:.1f} mm "
      f"(conceptual {fus_est['L_vane_arm_m']*1e3:.1f} mm)")
print(f"{'Control authority vs NB3 assumption':<34}: x{factor:.2f}")
assert factor >= 1.0, (
    "as-selected vane arm shorter than the NB3 sizing assumption -- "
    "re-visit L_CG_m in config")

# Section 7 — Output Export

`out/fuselage_cots.yaml` — same schema as `out/fuselage.yaml` plus the
`as_selected` mass rollup, the `bay_fit` report, and the
`delta_vs_conceptual` block (pinned by the regression tests). Consumed by
the design summary (NB15); **not** by the CAD/CFD chain, which stays on
the conceptual geometry until a config revision adopts these findings.

---

In [ ]:
from conceptual_design.fuselage_design import write_fuselage_yaml

delta_vs_conceptual = {
    "D_fus_mm":      round((fus.D_fus - fus_est["D_fus_m"]) * 1e3, 2),
    "L_fus_mm":      round((fus.L_fus - fus_est["L_fus_m"]) * 1e3, 2),
    "x_CG_mm":       round((fus.x_CG - fus_est["x_CG_m"]) * 1e3, 2),
    "S_wet_m2":      round(fus.S_wet_m2 - fus_est["S_wet_m2"], 5),
    "CD0_fus":       round(fus.CD0_fus - fus_est["CD0_fus"], 5),
    "L_vane_arm_mm": round((fus.L_vane_arm - fus_est["L_vane_arm_m"]) * 1e3, 2),
}

write_fuselage_yaml(
    fus, fus_p_cots, OUT_PATH / "fuselage_cots.yaml",
    regen_notebook="notebooks/fuselage_design_cots.ipynb",
    extra={
        "as_selected": {
            "m_items_total_kg":     round(m_asselected, 5),
            "m_closure_mtow_kg":    round(MTOW_kg, 5),
            "m_delta_kg":           round(m_delta, 5),
            "rho_battery_eff_kgm3": round(rho_batt_eff, 1),
            "battery_id": battery.id, "fc_id": fc.id, "servo_id": servo.id,
            "edf_motor_id": comps["edf_motor"].id,
            "esc_id": comps["esc"].id, "propeller_id": comps["propeller"].id,
        },
        "bay_fit": fit,
        "delta_vs_conceptual": delta_vs_conceptual,
    },
)
print(f"As-selected fuselage design written -> {OUT_PATH / 'fuselage_cots.yaml'}")

# Section 8 — Design Summary

---

In [ ]:
reports.print_fuselage_cots_summary(fus, fus_est, fus_p_cots, m_asselected,
                                    m_delta, fit)